# FFTW C2C F90 MPI Parallel Sequana X

* running on the login node sdumont18, without srun

## FFT 3D 500x500x500

In [1]:
%%writefile fc2cpx.f90
program main
    use, intrinsic :: iso_c_binding
    use MPI
    implicit none
    include 'fftw3-mpi.f03'
    integer :: mpirank, mpisize, mpierror
    integer(C_INTPTR_T), parameter :: N = 500, M = N, L = N
    type(C_PTR) :: plan, cdata
    complex(C_DOUBLE_COMPLEX), pointer :: data(:,:,:)
    integer(C_INTPTR_T) :: alloc_local, local_N, local_start
    complex(C_DOUBLE_COMPLEX) :: s, rs
    integer :: i, j, k
    double precision :: r, t0, t1, t2, t3

    call MPI_Init(mpierror)
    call MPI_Comm_rank(MPI_COMM_WORLD, mpirank, mpierror)
    call MPI_Comm_size(MPI_COMM_WORLD, mpisize, mpierror)
    if (mpirank == 0) then
        call cpu_time(t0)    ! <--- time measurement
    endif

    call fftw_mpi_init()    

    ! get local data size and allocate (note dimension reversal)
    alloc_local = fftw_mpi_local_size_3d(N, M, L,  &
                 MPI_COMM_WORLD, local_N, local_start)
    cdata = fftw_alloc_complex(alloc_local)
    call c_f_pointer(cdata, data, [L, M, local_N])

    ! create MPI plan for in-place forward DFT (note dimension reversal)
    plan = fftw_mpi_plan_dft_3d(N, M, L, data, data,  &
                MPI_COMM_WORLD, FFTW_FORWARD, FFTW_ESTIMATE)

    ! initialize data
    if (mpirank == 0) then
      do k = 1, local_N
        do j = 1, M
          do i = 1, L
            r = ((i-1)*M*L + (j-1)*M  + (k + local_start))*1E-6
            data(i, j, k) = cmplx(r, 0)
          enddo
        enddo
      enddo
    endif

    ! compute transform (as many times as desired)
    call fftw_mpi_execute_dft(plan, data, data)

    ! compute the checksum of processes
    s = sum(data)
    if (mpirank == 0) then
        call cpu_time(t2)    ! <--- time measurement
    endif
    call MPI_Reduce(s,        &! send data
                    rs,       &! recv data
                    1,        &! count
                    MPI_DOUBLE_COMPLEX,  &! data type
                    MPI_SUM,  &! operation
                    0,        &! rank of root process
                    MPI_COMM_WORLD, mpierror)
    if (mpirank == 0) then
        call cpu_time(t3)    ! <--- time measurement
    endif
    
    call fftw_destroy_plan(plan)
    call fftw_free(cdata)
    call fftw_mpi_cleanup()
    
    ! show the result
    if (mpirank == 0) then
        call cpu_time(t1)    ! <--- time measurement        
        write(*, "('S: 'sf0.4spf0.4'j')", advance="no") rs
        write(*, "('  T: 'sf0.4' s')", advance="no") t1-t0
        write(*, "('  Tc: 'sf0.4' s')", advance="no") t3-t2
        write(*, "('  N: 'g0)") mpisize
    endif
    
    call mpi_finalize(mpierror)
end

Writing fc2cpx.f90


## Compiling

In [9]:
%%bash
module load sequana/current
module load sdbase
module load openmpi/gnu/4.0.4_ucx_1.6
dir=/scratch/app/mathlibs/fftw/3.3.8_openmpi-3.1_gnu
mpif90 -O3 -o fc2cpx fc2cpx.f90  \
    -L $dir/lib  -l fftw3_mpi  -l fftw3  -l m  -I $dir/include

Loading SEQUANA Software environment
Loading SDumont BASE Software environment


In [13]:
! ls fc2cpx

fc2cpx


## Run (on sdumont18)

## 1

In [14]:
%%bash
# 1, 4, 9, 16
module load openmpi/gnu/4.0.4_ucx_1.6
mpiexec -n 1 ./fc2cpx

S: 125.0000+.0000j  T: 4.0408 s  Tc: .0001 s  N: 1


In [39]:
%%bash    
# 1, 4, 9, 16
module load openmpi/gnu/4.0.4_ucx_1.6
mpiexec -n 1 ./fc2cpx

S: 125.0000+.0000j  T: 4.0991 s  Tc: .0001 s  N: 1


In [40]:
%%bash    
# 1, 4, 9, 16
module load openmpi/gnu/4.0.4_ucx_1.6
mpiexec -n 1 ./fc2cpx

S: 125.0000+.0000j  T: 4.0329 s  Tc: .0001 s  N: 1


## 4

In [41]:
%%bash    
# 1, 4, 9, 16
module load openmpi/gnu/4.0.4_ucx_1.6
mpiexec -n 4 ./fc2cpx

S: 125.0000-.0000j  T: 1.6019 s  Tc: .0008 s  N: 4


In [42]:
%%bash    
# 1, 4, 9, 16
module load openmpi/gnu/4.0.4_ucx_1.6
mpiexec -n 4 ./fc2cpx

S: 125.0000-.0000j  T: 1.6135 s  Tc: .0011 s  N: 4


In [43]:
%%bash    
# 1, 4, 9, 16
module load openmpi/gnu/4.0.4_ucx_1.6
mpiexec -n 4 ./fc2cpx

S: 125.0000-.0000j  T: 1.6070 s  Tc: .0001 s  N: 4


## 9

In [44]:
%%bash    
# 1, 4, 9, 16
module load openmpi/gnu/4.0.4_ucx_1.6
mpiexec -n 9 ./fc2cpx

S: 125.0000+.0000j  T: 1.6716 s  Tc: .0006 s  N: 9


In [45]:
%%bash    
# 1, 4, 9, 16
module load openmpi/gnu/4.0.4_ucx_1.6
mpiexec -n 9 ./fc2cpx

S: 125.0000+.0000j  T: 1.6520 s  Tc: .0023 s  N: 9


In [46]:
%%bash    
# 1, 4, 9, 16
module load openmpi/gnu/4.0.4_ucx_1.6
mpiexec -n 9 ./fc2cpx

S: 125.0000+.0000j  T: 1.6720 s  Tc: .0001 s  N: 9


## 16

In [47]:
%%bash    
# 1, 4, 9, 16
module load openmpi/gnu/4.0.4_ucx_1.6
mpiexec -n 16 ./fc2cpx

S: 125.0000-.0000j  T: .5603 s  Tc: .0020 s  N: 16


In [48]:
%%bash    
# 1, 4, 9, 16
module load openmpi/gnu/4.0.4_ucx_1.6
mpiexec -n 16 ./fc2cpx

S: 125.0000-.0000j  T: .5605 s  Tc: .0013 s  N: 16


In [49]:
%%bash    
# 1, 4, 9, 16
module load openmpi/gnu/4.0.4_ucx_1.6
mpiexec -n 16 ./fc2cpx

S: 125.0000-.0000j  T: .5621 s  Tc: .0012 s  N: 16


## FFT 3D 500x500x500

In [50]:
%%writefile fc2cpx2.f90
program main
    use, intrinsic :: iso_c_binding
    use MPI
    implicit none
    include 'fftw3-mpi.f03'
    integer :: mpirank, mpisize, mpierror
    integer(C_INTPTR_T), parameter :: N = 1000, M = N, L = N
    type(C_PTR) :: plan, cdata
    complex(C_DOUBLE_COMPLEX), pointer :: data(:,:,:)
    integer(C_INTPTR_T) :: alloc_local, local_N, local_start
    complex(C_DOUBLE_COMPLEX) :: s, rs
    integer :: i, j, k
    double precision :: r, t0, t1, t2, t3

    call MPI_Init(mpierror)
    call MPI_Comm_rank(MPI_COMM_WORLD, mpirank, mpierror)
    call MPI_Comm_size(MPI_COMM_WORLD, mpisize, mpierror)
    if (mpirank == 0) then
        call cpu_time(t0)    ! <--- time measurement
    endif

    call fftw_mpi_init()    

    ! get local data size and allocate (note dimension reversal)
    alloc_local = fftw_mpi_local_size_3d(N, M, L,  &
                 MPI_COMM_WORLD, local_N, local_start)
    cdata = fftw_alloc_complex(alloc_local)
    call c_f_pointer(cdata, data, [L, M, local_N])

    ! create MPI plan for in-place forward DFT (note dimension reversal)
    plan = fftw_mpi_plan_dft_3d(N, M, L, data, data,  &
                MPI_COMM_WORLD, FFTW_FORWARD, FFTW_ESTIMATE)

    ! initialize data
    if (mpirank == 0) then
      do k = 1, local_N
        do j = 1, M
          do i = 1, L
            r = ((i-1)*M*L + (j-1)*M  + (k + local_start))*1E-6
            data(i, j, k) = cmplx(r, 0)
          enddo
        enddo
      enddo
    endif

    ! compute transform (as many times as desired)
    call fftw_mpi_execute_dft(plan, data, data)

    ! compute the checksum of processes
    s = sum(data)
    if (mpirank == 0) then
        call cpu_time(t2)    ! <--- time measurement
    endif
    call MPI_Reduce(s,        &! send data
                    rs,       &! recv data
                    1,        &! count
                    MPI_DOUBLE_COMPLEX,  &! data type
                    MPI_SUM,  &! operation
                    0,        &! rank of root process
                    MPI_COMM_WORLD, mpierror)
    if (mpirank == 0) then
        call cpu_time(t3)    ! <--- time measurement
    endif
    
    call fftw_destroy_plan(plan)
    call fftw_free(cdata)
    call fftw_mpi_cleanup()
    
    ! show the result
    if (mpirank == 0) then
        call cpu_time(t1)    ! <--- time measurement        
        write(*, "('S: 'sf0.4spf0.4'j')", advance="no") rs
        write(*, "('  T: 'sf0.4' s')", advance="no") t1-t0
        write(*, "('  Tc: 'sf0.4' s')", advance="no") t3-t2
        write(*, "('  N: 'g0)") mpisize
    endif
    
    call mpi_finalize(mpierror)
end

Writing fc2cpx2.f90


## Compiling

In [51]:
%%bash
module load openmpi/gnu/4.0.4_ucx_1.6
dir=/scratch/app/mathlibs/fftw/3.3.8_openmpi-3.1_gnu
mpif90 -O3 -o fc2cpx2 fc2cpx2.f90  \
    -L $dir/lib  -l fftw3_mpi  -l fftw3  -l m  -I $dir/include

In [53]:
! ls fc2cpx2

fc2cpx2


## Run (on sdumont18)

## 1

In [55]:
%%bash
# 1, 4, 9, 16
module load openmpi/gnu/4.0.4_ucx_1.6
mpiexec -n 1 ./fc2cpx2

S: 1000.0001-.0003j  T: 37.7596 s  Tc: .0001 s  N: 1


In [56]:
%%bash    
# 1, 4, 9, 16
module load openmpi/gnu/4.0.4_ucx_1.6
mpiexec -n 1 ./fc2cpx2

S: 1000.0001-.0003j  T: 37.7654 s  Tc: .0001 s  N: 1


In [58]:
%%bash    
# 1, 4, 9, 16
module load openmpi/gnu/4.0.4_ucx_1.6
mpiexec -n 1 ./fc2cpx2

S: 1000.0001-.0003j  T: 38.0698 s  Tc: .0001 s  N: 1


## 4

In [59]:
%%bash    
# 1, 4, 9, 16
module load openmpi/gnu/4.0.4_ucx_1.6
mpiexec -n 4 ./fc2cpx2

S: 1000.0006+.0002j  T: 12.9425 s  Tc: .1509 s  N: 4


In [60]:
%%bash    
# 1, 4, 9, 16
module load openmpi/gnu/4.0.4_ucx_1.6
mpiexec -n 4 ./fc2cpx2

S: 1000.0006+.0002j  T: 13.0393 s  Tc: .0418 s  N: 4


In [61]:
%%bash    
# 1, 4, 9, 16
module load openmpi/gnu/4.0.4_ucx_1.6
mpiexec -n 4 ./fc2cpx2

S: 1000.0006+.0002j  T: 12.8876 s  Tc: .0519 s  N: 4


## 9

In [62]:
%%bash    
# 1, 4, 9, 16
module load openmpi/gnu/4.0.4_ucx_1.6
mpiexec -n 9 ./fc2cpx2

S: 1000.0004+.0003j  T: 15.6877 s  Tc: .0001 s  N: 9


In [63]:
%%bash    
# 1, 4, 9, 16
module load openmpi/gnu/4.0.4_ucx_1.6
mpiexec -n 9 ./fc2cpx2

S: 1000.0004+.0003j  T: 15.8220 s  Tc: .0001 s  N: 9


In [64]:
%%bash    
# 1, 4, 9, 16
module load openmpi/gnu/4.0.4_ucx_1.6
mpiexec -n 9 ./fc2cpx2

S: 1000.0004+.0003j  T: 15.7410 s  Tc: .0236 s  N: 9


## 16

In [65]:
%%bash    
# 1, 4, 9, 16
module load openmpi/gnu/4.0.4_ucx_1.6
mpiexec -n 16 ./fc2cpx2

S: 1000.0003-.0000j  T: 4.1016 s  Tc: .0216 s  N: 16


In [66]:
%%bash    
# 1, 4, 9, 16
module load openmpi/gnu/4.0.4_ucx_1.6
mpiexec -n 16 ./fc2cpx2

S: 1000.0003-.0000j  T: 4.1381 s  Tc: .0001 s  N: 16


In [67]:
%%bash    
# 1, 4, 9, 16
module load openmpi/gnu/4.0.4_ucx_1.6
mpiexec -n 16 ./fc2cpx2

S: 1000.0003-.0000j  T: 4.0974 s  Tc: .0233 s  N: 16


## FFT 3D 1500x1500x1500

In [71]:
%%writefile fc2cpx3.f90
program main
    use, intrinsic :: iso_c_binding
    use MPI
    implicit none
    include 'fftw3-mpi.f03'
    integer :: mpirank, mpisize, mpierror
    integer(C_INTPTR_T), parameter :: N = 1500, M = N, L = N
    type(C_PTR) :: plan, cdata
    complex(C_DOUBLE_COMPLEX), pointer :: data(:,:,:)
    integer(C_INTPTR_T) :: alloc_local, local_N, local_start
    complex(C_DOUBLE_COMPLEX) :: s, rs
    integer :: i, j, k
    double precision :: r, t0, t1, t2, t3

    call MPI_Init(mpierror)
    call MPI_Comm_rank(MPI_COMM_WORLD, mpirank, mpierror)
    call MPI_Comm_size(MPI_COMM_WORLD, mpisize, mpierror)
    if (mpirank == 0) then
        call cpu_time(t0)    ! <--- time measurement
    endif

    call fftw_mpi_init()    

    ! get local data size and allocate (note dimension reversal)
    alloc_local = fftw_mpi_local_size_3d(N, M, L,  &
                 MPI_COMM_WORLD, local_N, local_start)
    cdata = fftw_alloc_complex(alloc_local)
    call c_f_pointer(cdata, data, [L, M, local_N])

    ! create MPI plan for in-place forward DFT (note dimension reversal)
    plan = fftw_mpi_plan_dft_3d(N, M, L, data, data,  &
                MPI_COMM_WORLD, FFTW_FORWARD, FFTW_ESTIMATE)

    ! initialize data
    if (mpirank == 0) then
      do k = 1, local_N
        do j = 1, M
          do i = 1, L
            r = ((i-1)*M*L + (j-1)*M  + (k + local_start))*1E-6
            data(i, j, k) = cmplx(r, 0)
          enddo
        enddo
      enddo
    endif

    ! compute transform (as many times as desired)
    call fftw_mpi_execute_dft(plan, data, data)

    ! compute the checksum of processes
    s = sum(data)
    if (mpirank == 0) then
        call cpu_time(t2)    ! <--- time measurement
    endif
    call MPI_Reduce(s,        &! send data
                    rs,       &! recv data
                    1,        &! count
                    MPI_DOUBLE_COMPLEX,  &! data type
                    MPI_SUM,  &! operation
                    0,        &! rank of root process
                    MPI_COMM_WORLD, mpierror)
    if (mpirank == 0) then
        call cpu_time(t3)    ! <--- time measurement
    endif
    
    call fftw_destroy_plan(plan)
    call fftw_free(cdata)
    call fftw_mpi_cleanup()
    
    ! show the result
    if (mpirank == 0) then
        call cpu_time(t1)    ! <--- time measurement        
        write(*, "('S: 'sf0.4spf0.4'j')", advance="no") rs
        write(*, "('  T: 'sf0.4' s')", advance="no") t1-t0
        write(*, "('  Tc: 'sf0.4' s')", advance="no") t3-t2
        write(*, "('  N: 'g0)") mpisize
    endif
    
    call mpi_finalize(mpierror)
end

Overwriting fc2cpx3.f90


## Compiling

In [72]:
%%bash
module load openmpi/gnu/4.0.4_ucx_1.6
dir=/scratch/app/mathlibs/fftw/3.3.8_openmpi-3.1_gnu
mpif90 -O3 -o fc2cpx3 fc2cpx3.f90  \
    -L $dir/lib  -l fftw3_mpi  -l fftw3  -l m  -I $dir/include

In [73]:
! ls fc2cpx3

fc2cpx3


## Run (on sdumont18)

## 1

In [75]:
%%bash
# 1, 4, 9, 16
module load openmpi/gnu/4.0.4_ucx_1.6
mpiexec -n 1 ./fc2cpx3

S: 3375.0019+.0031j  T: 117.5158 s  Tc: .0001 s  N: 1


## 4

In [76]:
%%bash    
# 1, 4, 9, 16
module load openmpi/gnu/4.0.4_ucx_1.6
mpiexec -n 4 ./fc2cpx3

S: 3374.9972+.0040j  T: 43.9095 s  Tc: .2363 s  N: 4


## 9

In [77]:
%%bash    
# 1, 4, 9, 16
module load openmpi/gnu/4.0.4_ucx_1.6
mpiexec -n 9 ./fc2cpx3

S: 3375.0056-.0011j  T: 22.9068 s  Tc: .1589 s  N: 9


## 16

In [78]:
%%bash    
# 1, 4, 9, 16
module load openmpi/gnu/4.0.4_ucx_1.6
mpiexec -n 16 ./fc2cpx3

S: 3374.9989+.0008j  T: 13.8469 s  Tc: .0982 s  N: 16


## 36

In [79]:
%%bash    
# 1, 4, 9, 16
module load openmpi/gnu/4.0.4_ucx_1.6
mpiexec -n 36 ./fc2cpx3

S: 3375.0039-.0029j  T: 8.4479 s  Tc: .0473 s  N: 36
